<a href="https://colab.research.google.com/github/swaneedh/machine-learing/blob/main/exp12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

from sklearn.metrics import (
   accuracy_score,
   precision_score,
   recall_score,
   f1_score
)

# Load Titanic dataset from a public URL instead of a local file
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
df = pd.read_csv(url)

# Map columns to match the standard Titanic casing
df = df.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked"
})

# Select features
features = [
   "Pclass", "Sex", "Age", "SibSp",
   "Parch", "Fare", "Embarked"
]

X = df[features]
y = df["Survived"]

# Feature types
numeric_features = [
   "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]

# Preprocessing pipeline with imputation for handling missing values
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
   transformers=[
       ("num", numeric_transformer, numeric_features),
       ("cat", categorical_transformer, categorical_features)
   ]
)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2,
   random_state=42, stratify=y
)

# Base Decision Tree
base_model = DecisionTreeClassifier(
   max_depth=4,
   random_state=42
)

# Bagging
bagging = BaggingClassifier(
   estimator=base_model,
   n_estimators=50,
   random_state=42
)

bagging_pipeline = Pipeline([
   ("preprocessor", preprocessor),
   ("classifier", bagging)
])

bagging_pipeline.fit(X_train, y_train)
bag_pred = bagging_pipeline.predict(X_test)

# AdaBoost
boosting = AdaBoostClassifier(
   estimator=DecisionTreeClassifier(
       max_depth=2,
       random_state=42
   ),
   n_estimators=50,
   random_state=42
)

boosting_pipeline = Pipeline([
   ("preprocessor", preprocessor),
   ("classifier", boosting)
])

boosting_pipeline.fit(X_train, y_train)
boost_pred = boosting_pipeline.predict(X_test)

# Evaluation function
def evaluate_model(name, y_true, y_pred):
   print("\n", name)

   print("Accuracy:",
         round(accuracy_score(y_true, y_pred), 4))

   print("Precision:",
         round(precision_score(y_true, y_pred), 4))

   print("Recall:",
         round(recall_score(y_true, y_pred), 4))

   print("F1-score:",
         round(f1_score(y_true, y_pred), 4))

# Evaluate
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("Boosting (AdaBoost)", y_test, boost_pred)


 Bagging
Accuracy: 0.8045
Precision: 0.8696
Recall: 0.5797
F1-score: 0.6957

 Boosting (AdaBoost)
Accuracy: 0.7765
Precision: 0.7636
Recall: 0.6087
F1-score: 0.6774
